# NTGEN: General Nanotube Generation from the Alexandria 1D Database

**Google Colab notebook — requires a GPU runtime.**
Before running: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

This notebook generates new general (multi-element) nanotube candidates with the
NTGEN diffusion model, seeded by **real 1D nanotube structures** from the
Alexandria database. It uses SCIGEN-style mask-constrained denoising
("Pathway 3") — the pretrained mp_20 diffusion model is used as-is, **no
retraining required**.

**Pipeline**: sample a real nanotube template from
`data/nano_1D/nanotube_templates.npz` (2,216 structures distilled from the 7,002
ASE structures in `alexandria_1d_nanotubes.pkl`) → `SC_DBShell` keeps only the
template's *geometry* (cell + a transverse radial band `[r_min, r_max]`) and pins
**no atoms** → `sample_scigen` reverse diffusion generates every atom's position
and species from noise, softly confined to the wall shell at each step →
`pymatgen` → CIF.

> **Opening this notebook in Colab**: once the repo is public, open
> [colab.research.google.com/github/3venthatguy/NTU-IQM/blob/main/comp_models/NTGEN_generation/ntgen_generation.ipynb](https://colab.research.google.com/github/3venthatguy/NTU-IQM/blob/main/comp_models/NTGEN_generation/ntgen_generation.ipynb)
> — or in Colab use **File → Open notebook → GitHub** and paste the repo URL.


In [ ]:
# takes long (~5-10 min first run) — clones repo, installs deps, downloads model

# --- Run once per Colab session ---
import os, shutil, subprocess, sys

REPO_URL = 'https://github.com/3venthatguy/NTU-IQM.git'
REPO_DIR = '/content/NTU-IQM'
PROJECT_DIR = f'{REPO_DIR}/NTGENS'

# Clone repo if needed (the repo must be public for an anonymous clone)
if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])

# The package dir is named ntgent/ but all imports & hydra targets say scigen.*
# -> self-heal with a symlink.
if not os.path.exists(f'{PROJECT_DIR}/scigen') and os.path.exists(f'{PROJECT_DIR}/ntgent'):
    os.symlink('ntgent', f'{PROJECT_DIR}/scigen')
    print('created symlink scigen -> ntgent')

# Install PyG extensions (need matching torch+CUDA wheels)
try:
    import torch_scatter
except ImportError:
    import torch
    _vp = torch.__version__.split('+')[0].split('.')
    tv = f'{_vp[0]}.{_vp[1]}.0'
    if torch.version.cuda:
        cv = 'cu' + torch.version.cuda.replace('.', '')
    else:
        cv = 'cpu'
    whl = f'https://data.pyg.org/whl/torch-{tv}+{cv}.html'
    print(f'Installing PyG extensions from: {whl}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'torch-scatter', 'torch-sparse', 'torch-cluster',
                           '-f', whl])

# Remaining dependencies (torch itself comes preinstalled on Colab)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'hydra-core', 'omegaconf', 'pytorch-lightning',
                       'pymatgen', 'torch-geometric', 'einops',
                       'p_tqdm', 'pyxtal', 'pathos', 'python-dotenv',
                       'scipy', 'scikit-learn', 'networkx'])

# Download the pretrained SCIGEN mp_20 checkpoint from Figshare if not present
from pathlib import Path
MODEL_PATH = Path(PROJECT_DIR) / 'models' / 'mp_20'
if not MODEL_PATH.exists() or not list(MODEL_PATH.glob('*.ckpt')):
    import json, zipfile
    from urllib.request import urlopen, urlretrieve
    MODEL_PATH.mkdir(parents=True, exist_ok=True)
    article = json.loads(urlopen('https://api.figshare.com/v2/articles/27778134').read().decode())
    for f in article.get('files', []):
        dest = MODEL_PATH / f['name']
        print(f'Downloading {f["name"]}...')
        urlretrieve(f['download_url'], str(dest))
        if dest.suffix == '.zip':
            with zipfile.ZipFile(dest, 'r') as zf:
                zf.extractall(MODEL_PATH)
            dest.unlink()

os.environ.setdefault('PROJECT_ROOT', PROJECT_DIR)
print('Ready.')


---
## 1. How NTGEN works

NTGEN is the SCIGEN diffusion framework focused purely on **nanotube
generation**. The diffusion model generates a complete crystal (lattice + atom
positions + species) from noise; at every denoising step a soft constraint mask
steers the result toward a nanotube by **confining every atom to the tube's wall
shell** (`shl`, pin only the geometry and generate all atoms fresh).

### Available structural constraints (`sc_dict` in `script/sc_utils.py`)

| Code | Constraint | Known atoms | Description |
|------|------------|-------------|-------------|
| `shl` | **Geometric shell** | none (0) | Pin only a real tube's *shape* (cell + radial band `[r_min, r_max]`); generate **all** atoms fresh, softly confined to the wall shell |
| `van` | Vanilla | 1 | Unconstrained generation (baseline) |

> Earlier atom-pinning modes (`ntb` parametric ring, `cnt` carbon wall, `alx`
> real structure pinned atom-by-atom) were **removed**: they pinned most or all
> atoms, leaving nothing for the model to generate. `shl` keeps only the shape.

> **Dataset note**: run `'shl'` with a *general* dataset (`mp_20`). `carbon_24`
> would force every atom to carbon and flatten the multi-element chemistry.
>
> **`shl` scale caveat**: a geometric-shell candidate generates the tube's full
> real atom count (median ~42), which is out-of-distribution for the ≤20-atom
> `mp_20` checkpoint — the *shape* comes out tube-faithful, but good in-shell
> *chemistry* needs an Alexandria-trained checkpoint (see `RETRAIN_ALX.md`).

---
## 2. Environment setup

Safe to re-run at any time (e.g. after a runtime restart — the clone and
installs from the setup cell above persist for the session).


In [ ]:
import os, sys

PROJECT_DIR = os.environ.get('PROJECT_ROOT', '/content/NTU-IQM/NTGENS')
NOTEBOOK_DIR = '/content/NTU-IQM/comp_models/NTGEN_generation'

assert os.path.exists(PROJECT_DIR), (
    f'{PROJECT_DIR} not found — run the setup cell at the top first.')

# scigen -> ntgent symlink (self-heal if the setup cell was skipped)
if not os.path.exists(f'{PROJECT_DIR}/scigen') and os.path.exists(f'{PROJECT_DIR}/ntgent'):
    os.symlink('ntgent', f'{PROJECT_DIR}/scigen')

os.environ.setdefault('PROJECT_ROOT', PROJECT_DIR)
os.environ.setdefault('HYDRA_JOBS', PROJECT_DIR)
os.environ.setdefault('WANDB_DIR', os.path.join(PROJECT_DIR, 'wandb'))
os.environ.setdefault('WANDB_MODE', 'disabled')   # no wandb login prompts
os.makedirs(os.environ['WANDB_DIR'], exist_ok=True)

for p in (PROJECT_DIR, os.path.join(PROJECT_DIR, 'script')):
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(PROJECT_DIR)   # gen_utils opens ./data/... relative to cwd

# The Alexandria template cache ships with the repo (648 KB, built once from
# the raw 48 MB ASE pickle by data/nano_1D/build_templates.py).
cache = os.path.join(PROJECT_DIR, 'data', 'nano_1D', 'nanotube_templates.npz')
assert os.path.exists(cache), (
    f'{cache} missing — the clone is incomplete or the cache was never '
    'committed. Re-run the setup cell, or rebuild with '
    'data/nano_1D/build_templates.py (requires the raw pickle).')

print(f'Working directory: {os.getcwd()}')
print(f'Template cache:    {cache} ({os.path.getsize(cache)/1024:.0f} KB)')


---
## 3. Load the pretrained model

We load the SCIGEN mp_20 diffusion model using Hydra for configuration and
manual state-dict loading for compatibility with any PyTorch Lightning version,
then attach the constraint-aware sampler `sample_scigen`.


In [ ]:
import torch
import numpy as np
import hydra
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from pathlib import Path

import scigen
print(f'scigen imported from: {scigen.__path__[0]}')


def load_model_for_inference(model_path, device='cpu'):
    """Load the SCIGEN pretrained model for inference.

    Uses manual state_dict loading instead of pytorch_lightning's
    load_from_checkpoint, so it works with any PL version.
    """
    model_path = Path(model_path)

    # Clear previous hydra state (allows re-running this cell)
    GlobalHydra.instance().clear()

    # Load model config from hparams.yaml
    with initialize_config_dir(config_dir=str(model_path.resolve()), version_base=None):
        cfg = compose(config_name='hparams')

    # Instantiate model architecture (empty weights)
    model = hydra.utils.instantiate(
        cfg.model, optim=cfg.optim, data=cfg.data,
        logging=cfg.logging, _recursive_=False,
    )

    # Find checkpoint file
    ckpts = sorted(model_path.glob('*.ckpt'))
    if not ckpts:
        raise FileNotFoundError(f'No .ckpt files found in {model_path}')

    ckpt_path = next((c for c in ckpts if 'last' in c.name), ckpts[-1])
    print(f'Loading checkpoint: {ckpt_path.name}')

    checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['state_dict'], strict=False)

    # Load data scalers
    for attr, fname in [('lattice_scaler', 'lattice_scaler.pt'),
                        ('scaler', 'prop_scaler.pt')]:
        fpath = model_path / fname
        if fpath.exists():
            setattr(model, attr, torch.load(fpath, map_location='cpu', weights_only=False))

    model = model.to(device)
    model.eval()
    return model, cfg


In [ ]:
MODEL_PATH = Path(PROJECT_DIR) / 'models' / 'mp_20'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type != 'cuda':
    print('WARNING: no GPU detected — generation will be very slow. '
          'Use Runtime -> Change runtime type -> T4 GPU.')

model, cfg = load_model_for_inference(MODEL_PATH, device=device)

# Attach the constraint-aware sampling method
from scigen.pl_modules.diffusion_w_type import sample_scigen
model.sample_scigen = sample_scigen.__get__(model)

num_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded: {num_params:,} parameters')


---
## 4. Generation settings

- `SC_TYPE='shl'` (**geometric shell**): draw a real tube but pin **only its
  shape** — the cell (tube axis, axial period, vacuum box) and a transverse
  **radial band** `[r_min, r_max]` (the wall shell, from the Alexandria
  forensics). **No atoms are pinned** (`num_known=0`); the model generates every
  atom's position *and* species from noise, and the soft radial mask confines
  them onto the shell as `ψ(t)` → 1. This is de-novo generation *inside a real
  tube envelope* — maximal decoration freedom, tube-enforced geometry. If no
  template fits the range, that draw silently falls back to the private
  synthetic ring (`_SC_NanotubeFallback`).
- `DATASET='mp_20'` supplies the atom-count distribution (up to 20 atoms) and
  keeps atom types free for the diffusion model — required for multi-element
  tubes.
- `KNOWN_SPECIES` only feeds the bond-length sampler; `SC_DBShell` ignores it
  (it carries its own geometry).
- `reduced_mask=False` is required: atoms are constrained per-coordinate
  (`mask_x` has shape `(N, 3)`).

### Constrained-sampling controls (new)

Optional, **no-retraining** refinements read by `sample_scigen` off runtime
attributes (`model.pin_cfg` / `model.cyl_cfg` / `model.dens_cfg`, set in Section
5), so they need no change to the frozen checkpoint. Set `PIN_SCHEDULE='none'` and `CYL_MASKING = DENSITY_MASKING = False`
to reproduce the original binary sampler exactly.

- **Progressive skeleton pinning** (`PIN_SCHEDULE`): instead of freezing the
  skeleton from `t=T` (a step-function mask the model never saw in training),
  ramp the pinning strength `ψ(t)` from 0 at `t=T` (whole cell starts as noise,
  *in-distribution*) to 1 near `t=0`. `linear` is a steady ramp; `sigmoid` is
  slow-early / fast-mid / slow-late (nucleate → grow → anneal). `'none'` =
  original binary pinning. `ψ(t)` also scales the shell confinement below.
- **Cylindrical radial band** (`CYL_MASKING`): softly keep atoms within the
  tube's transverse wall band. For `shl` it is **two-sided** — every atom is
  confined into `[r_min, r_max]` (`CYL_R_LO_PCT` sets the inner edge percentile;
  `r_max` is the template's `r₉₅`, from
  `comp_models/Analysis/nanotube_rtheta_forensics.ipynb`). Strength rises as
  `ψ(t)` → 1; atoms already in-band are untouched.
- **Frame tracking** (`TRACK_FRAME`, v3) — **the fix for atoms bunching on one arc.**
  The cell is *itself* diffused: `l_T` is pure noise (~1 Å) and only grows into the real
  template cell (~14 Å) as `C0 = √ᾱ` rises. The tube frame, though, was frozen in
  absolute template Ångströms — so for the first ~40 % of steps its centroid sat several
  cell-widths outside the atoms, every atom shared essentially **one azimuth**, and
  because the two radial terms below are *θ-invariant* (they only rescale `r`) that arc
  was slid onto the wall and locked in. Rebuilding the frame from the **current** lattice
  each step, with the band scaled by the transverse cell scale `s_t`, keeps the
  constraint self-consistent at every noise level. `s_t → 1` at `t=0`, so this is exactly
  the old behaviour there. Measured over 50 real templates: the circular resultant
  `R` (0 = spread, 1 = one arc) went `0.999 → 0.88` across early steps with the fixed
  frame, versus a flat `~0.29` (the finite-`N` random floor) with tracking.

- **Angular dispersion** (`ANG_SPREAD`, v3) — optional safety net. Descends
  `Σ_m w_m R̄_m²` where `R̄_m` is the *m*-th circular resultant, pushing atoms away from
  the resultant angle. Self-limiting (the drive vanishes at uniformity) and
  rotation-invariant. Off by default: frame tracking alone already reaches the random
  floor. Mode 1 is the right default — real templates measure `R₁ ≈ 0.01`, so driving it
  to 0 matches ground truth, whereas `R₂ ≈ 0.10` is **genuine n-fold symmetry that must
  not be optimised away** (hence `ANG_MODE2_FLR`).

- **Decoupled geometry ramp** (`GEOM_PIN_SCHEDULE`, v3): `ψ(t)` does *double duty* — it
  blends the pinned lattice **and** scales the geometric forces. That is why raising
  `PSI_START` backfires: it runs the full-strength band against the noise-scale cell at
  `t=T`. Set this instead to ramp the forces independently of the mask.

- **Radial density guidance** (`DENSITY_MASKING`, v2): the band alone only
  *bounds* atoms — inside it they fill the whole disk, so a `shl` tube can look
  like a filled slab. This adds a soft **log-density gradient** force: it builds
  the template's own radial distribution ρ(r) (a Gaussian KDE of its atom radii)
  and nudges every atom *up* `∇_r log ρ(r)`, concentrating them onto the real
  **wall** (the mode of ρ) instead of the disk — turning the filled cross-section
  into a thin wall. `BANDWIDTH_SCALE` is the wall thickness (empirical per
  template; **<1 sharpens** the wall); a double-wall ρ(r) gives two attractors
  automatically. Also `ψ(t)`-scaled, layered on top of the band.

In [ ]:
# ============================================================
#  EDIT THESE PARAMETERS to try different generation settings
# ============================================================
SC_TYPE       = 'shl'    # 'shl' geometric shell (real tube shape, all atoms generated) | 'van' unconstrained
DATASET       = 'mp_20'  # general dataset: keeps atom species free for multi-element tubes
KNOWN_SPECIES = ['Fe']   # bond-length sampler only; ignored by 'shl'
BATCH_SIZE    = 4        # structures per batch (keep small for Colab T4)
NUM_BATCHES   = 1        # number of batches
# ============================================================

FRAC_Z  = 0.5            # unused by 'shl' (template sets geometry)
STEP_LR = 5e-6           # diffusion step size
SEED    = 42

# Atom count ranges per structural constraint.
# 'shl' admits REAL Alexandria tubes. With [1,20] only ~17% of the DB was drawable
# (tiny 2-3 atom fragments); [24,64] covers whole median-scale (~42-atom) tubes.
# Caveat: the mp_20 checkpoint was trained on <=20 atoms, so a 40+-atom cell is OOD for
# the MODEL -- geometry renders faithfully but good CHEMISTRY needs an Alexandria-trained
# checkpoint (see RETRAIN_ALX.md).
SC_NATM_RANGE = {'shl': [4, 128], 'van': [1, 20]}

# (#4) Plausible-element policy for decoded species: drop radioactive / synthetic
# elements that never form stable inorganic compounds -- At, Rn, Fr, Ra and all
# transuranics (incl. Fm = Z100). Applied at decode so generated atoms can't emerge as
# Fm/At.
DECORATOR_DISALLOWED_Z = set(range(85, 89)) | set(range(93, 101))  # At,Rn,Fr,Ra + Np..Fm

# ------------------------------------------------------------
#  Constrained-sampling controls (see Section 4). These are all ON
#  below; set PIN_SCHEDULE='none' + CYL_MASKING=DENSITY_MASKING=False
#  to fall back to the original binary-pinning sampler.
# ------------------------------------------------------------
# Progressive skeleton pinning: 'none' (binary) | 'linear' | 'sigmoid'.
PIN_SCHEDULE = 'sigmoid'
PIN_ALPHA    = 10.0      # sigmoid steepness
PIN_TMID     = 0.6       # sigmoid inflection as a fraction of T (1000 steps)
PSI_START    = 0.0       # pinning strength at t=T (0 -> start from noise)
PSI_END      = 1.0       # pinning strength at t=0 (1 -> fully enforced)
# !! KEEP PSI_START = 0.0 !!  psi(t) does DOUBLE DUTY: it blends the pinned lattice
# AND scales the geometric forces (strength = max_strength * psi * gate). Raising
# PSI_START runs the FULL-strength radial band against the noise-scale (~1 A) cell at
# t=T and blows structures apart (measured: pct_in 100% -> 67%). To hard-pin the
# lattice from t=T, leave PSI_START at 0.0 and use GEOM_PIN_SCHEDULE below, which
# ramps the forces independently of the mask.
# Cylindrical radial band. 'shl': two-sided [r_min, r_max] confining ALL atoms
# onto the wall shell.
CYL_MASKING  = True      # False -> atoms diffuse in the whole box (original)
CYL_MARGIN   = 0.1       # r_hi = r_max * (1 + margin)
CYL_R_LO_PCT = 5.0       # inner band-edge percentile
CYL_STRENGTH = 1.0       # peak radial pull (scaled by psi(t) each step)
# (v2) Radial density guidance: concentrate atoms onto the WALL within the band
# (fixes the "filled disk looks like a crystal" problem).
DENSITY_MASKING  = True  # False -> band only (atoms fill the disk uniformly)
DENSITY_STRENGTH = 0.05  # radial step scale of the log-density force (x psi(t))
BANDWIDTH_SCALE  = 1.0   # empirical wall thickness; <1 = thinner wall (try 0.4-0.6)
# (v3) Frame tracking -- THE fix for atoms bunching on one arc of the wall. The cell
# starts as pure noise (l_T ~ N(0,1), ~1 A) and only grows into the real template cell
# (~14 A) as the trajectory proceeds, but the tube frame was frozen in absolute
# template Angstroms: for the first ~40% of steps its centroid sat several cell-widths
# away, so every atom shared essentially ONE azimuth -- and since the radial band and
# density force are theta-invariant (they only rescale r), that arc got slid onto the
# wall and locked in. Rebuilding the frame from the CURRENT lattice each step, with the
# band scaled by the transverse cell scale s_t, keeps the constraint self-consistent at
# every noise level. s_t -> 1 at t=0, where this is exactly the old behaviour.
TRACK_FRAME   = True     # False -> legacy fixed frame (reproduces the bug)
CYL_SCALE_LO  = 0.25     # clamp s_t to this fraction of the linear cell-scale ratio
CYL_SCALE_HI  = 4.0      # ... and this multiple (guards near-degenerate noise cells)
# (v3) Angular dispersion -- optional safety net. Descends the circular resultant
# R (0 = uniform around the tube, 1 = all atoms at one angle). Off by default: frame
# tracking alone already brings R down to the finite-N random floor.
ANG_SPREAD    = False    # True -> also push atoms apart in theta
ANG_STRENGTH  = 0.05     # max radians of rotation per application (x psi(t))
ANG_MODES     = 1        # 1 = first Fourier moment (real templates sit at R1 ~ 0.01);
                         # 2 also splits antipodal pairs, which mode 1 cannot see
ANG_MODE2_W   = 0.25     # weight of mode 2 (only used when ANG_MODES >= 2)
ANG_MODE2_FLR = 0.10     # mode-2 deadband -- real tubes have genuine n-fold structure
                         # at R2 ~ 0.10, so do NOT drive it to zero
# (v3) Ramp for the GEOMETRIC forces, decoupled from the skeleton/lattice ramp above.
# 'same' -> reuse PIN_SCHEDULE (default, unchanged behaviour). Setting this lets you
# hard-pin the lattice (PSI_START=1.0) while the forces still ramp from 0.
GEOM_PIN_SCHEDULE = 'same'   # 'same' | 'none' | 'linear' | 'sigmoid'
GEOM_PSI_START    = 0.0
GEOM_PSI_END      = 1.0

natm_range = SC_NATM_RANGE.get(SC_TYPE, [1, 20])
total_structures = BATCH_SIZE * NUM_BATCHES

print('Generation settings:')
print(f'  Constraint:     {SC_TYPE}')
print(f'  Dataset:        {DATASET}')
print(f'  Structures:     {total_structures}')
print(f'  Atoms per cell: {natm_range[0]}-{natm_range[1]}' + (' (shl: = template nsites)' if SC_TYPE == 'shl' else ''))
print(f'  Pinning:        {PIN_SCHEDULE}' + (f' (alpha={PIN_ALPHA}, t_mid={PIN_TMID})' if PIN_SCHEDULE == 'sigmoid' else ''))
_band = 'two-sided [r_min, r_max]' if SC_TYPE == 'shl' else 'n/a'
print(f'  Cyl band:       {"on" if CYL_MASKING else "off"}' + (f' ({_band}, margin={CYL_MARGIN}, strength={CYL_STRENGTH})' if CYL_MASKING else ''))
print(f'  Frame tracking: {"on" if TRACK_FRAME else "OFF (legacy fixed frame)"}')
print(f'  Angular spread: {"on" if ANG_SPREAD else "off"}'
      + (f' (strength={ANG_STRENGTH}, modes={ANG_MODES})' if ANG_SPREAD else ''))
print(f'  Geom ramp:      {GEOM_PIN_SCHEDULE}'
      + ('' if GEOM_PIN_SCHEDULE == 'same' else f' (psi {GEOM_PSI_START} -> {GEOM_PSI_END})'))
print(f'  Density guide:  {"on" if DENSITY_MASKING else "off"}' + (f' (strength={DENSITY_STRENGTH}, bandwidth_scale={BANDWIDTH_SCALE})' if DENSITY_MASKING else ''))

---
## 5. Generate

`SampleDataset` draws one Alexandria template per candidate and builds the
constraint masks; the loop below runs reverse diffusion with the template
re-imposed at every step. The printout before generation shows exactly which
skeleton was pinned for each candidate.


In [ ]:
# takes long (~20-60 sec on a T4) -- runs diffusion sampling

import time
from tqdm.auto import tqdm
from torch_geometric.data import DataLoader
from gen_utils import SampleDataset
from sc_utils import chemical_symbols

# (#4) Allowed-species mask over the MAX_ATOMIC_NUM=100 type classes (class c -> Z=c+1).
# Disallowed Z are forced off before argmax so decoded species stay chemically plausible.
from scigen.pl_modules.diffusion_w_type import MAX_ATOMIC_NUM
ALLOWED_Z_MASK = torch.ones(MAX_ATOMIC_NUM, dtype=torch.bool)
for _z in DECORATOR_DISALLOWED_Z:
    if 1 <= _z <= MAX_ATOMIC_NUM:
        ALLOWED_Z_MASK[_z - 1] = False

# Constrained-sampling controls -> runtime attributes read by sample_scigen.
# (Set here, after Section 4's knobs are defined; the frozen checkpoint's
# hparams are not used for these.) Defaults reproduce the original binary sampler.
model.pin_cfg = {
    'enabled': PIN_SCHEDULE != 'none',
    'schedule': PIN_SCHEDULE,
    'alpha': PIN_ALPHA,
    't_mid_frac': PIN_TMID,
    'psi_start': PSI_START,
    'psi_end': PSI_END,
}
model.cyl_cfg = {
    'enabled': bool(CYL_MASKING),
    'margin': CYL_MARGIN,
    'r_lo_percentile': CYL_R_LO_PCT,
    'max_strength': CYL_STRENGTH,
    'track_frame': bool(TRACK_FRAME),    # v3: frame from the CURRENT lattice
    'scale_lo': CYL_SCALE_LO,
    'scale_hi': CYL_SCALE_HI,
    'ang_min': 0.05,
}
model.dens_cfg = {                       # v2 radial density guidance
    'enabled': bool(DENSITY_MASKING),
    'density_strength': DENSITY_STRENGTH,
    'bandwidth_scale': BANDWIDTH_SCALE,
    'grid_size': 96,
}
model.ang_cfg = {                        # v3 angular dispersion
    'enabled': bool(ANG_SPREAD),
    'ang_strength': ANG_STRENGTH,
    'mode_weights': (1.0,) if ANG_MODES < 2 else (1.0, ANG_MODE2_W),
    'r_floors':     (0.0,) if ANG_MODES < 2 else (0.0, ANG_MODE2_FLR),
    'min_atoms': 3,
    'max_dtheta': 0.5,
}
# Separate ramp for the geometric forces; None -> follow pin_cfg (unchanged).
model.geom_pin_cfg = None if GEOM_PIN_SCHEDULE == 'same' else {
    'enabled': GEOM_PIN_SCHEDULE != 'none',
    'schedule': GEOM_PIN_SCHEDULE,
    'alpha': PIN_ALPHA,
    't_mid_frac': PIN_TMID,
    'psi_start': GEOM_PSI_START,
    'psi_end': GEOM_PSI_END,
}
print('pin_cfg:', model.pin_cfg)
print('cyl_cfg:', model.cyl_cfg)
print('dens_cfg:', model.dens_cfg)
print('ang_cfg:', model.ang_cfg)
print('geom_pin_cfg:', model.geom_pin_cfg)

# Build the sample dataset with the nanotube constraint
test_set = SampleDataset(
    dataset=DATASET,
    natm_range=natm_range,
    total_num=total_structures,
    bond_sigma_per_mu=None,
    use_min_bond_len=False,
    known_species=KNOWN_SPECIES,
    sc_list=[SC_TYPE],
    frac_z=FRAC_Z,
    c_vec_cons={'scale': None, 'vert': False},
    reduced_mask=False,
    seed=SEED,
    device=device,
    max_decorators=None,                                         # 'shl' uses the tube's real nsites
    r_lo_percentile=CYL_R_LO_PCT,                                  # shl inner band edge
    bandwidth_scale=BANDWIDTH_SCALE,                              # v2 wall thickness
    density_grid_size=96,                                         # v2 force-table resolution
)

# Peek at each candidate: shell geometry ('shl') or unconstrained ('van')
print('Per-candidate constraint:')
for i in range(len(test_set)):
    d = test_set[i]
    K = int(d.num_known)
    if SC_TYPE == 'shl' and float(d.is_alx) > 0:
        # No atoms pinned -> report the wall band the model must fill.
        print(f'  [{i}] shell band r=[{float(d.r_min):.2f}, {float(d.r_max):.2f}] A, '
              f'axis={int(d.tube_axis)} -> generate {int(d.num_atoms[0])} atoms (num_known=0)')
    else:
        species = sorted({chemical_symbols[int(z)] for z in d.atom_types_known[:K]})
        envelope = f', r_max={float(d.r_max):.2f} A' if float(d.is_alx) > 0 else ''
        print(f'  [{i}] {K} pinned atoms {species} -> {int(d.num_atoms[0])} total{envelope}')

test_loader = DataLoader(test_set, batch_size=BATCH_SIZE)

# Run diffusion sampling
all_frac_coords, all_atom_types, all_lattices = [], [], []
all_num_atoms, all_num_known = [], []

print(f'\nRunning diffusion (step_lr={STEP_LR})...')
start_time = time.time()

for idx, batch in enumerate(tqdm(test_loader, desc='Generating structures')):
    if torch.cuda.is_available():
        batch.cuda()
    outputs, traj = model.sample_scigen(batch, step_lr=STEP_LR)

    all_frac_coords.append(outputs['frac_coords'].detach().cpu())
    raw_types = outputs['atom_types'].detach().cpu()
    if raw_types.dim() == 2:
        # (#4) forbid implausible species, then decode logits -> atomic numbers
        raw_types = raw_types.masked_fill(~ALLOWED_Z_MASK, float('-inf'))
        raw_types = raw_types.argmax(dim=-1) + 1
    all_atom_types.append(raw_types)
    all_lattices.append(outputs['lattices'].detach().cpu())
    all_num_atoms.append(outputs['num_atoms'].detach().cpu())
    all_num_known.append(outputs['num_known'].detach().cpu())

    print(f'  Batch {idx + 1}/{len(test_loader)} complete')

elapsed = time.time() - start_time
print(f'\nGeneration complete in {elapsed:.1f}s ({elapsed / total_structures:.1f}s per structure)')

frac_coords = torch.cat(all_frac_coords, dim=0)
atom_types = torch.cat(all_atom_types, dim=0)
lattices = torch.cat(all_lattices, dim=0)
num_atoms = torch.cat(all_num_atoms, dim=0)
num_known = torch.cat(all_num_known, dim=0)
print('Atoms per structure:', num_atoms.tolist(), '| known/pinned:', num_known.tolist())

In [ ]:
# --- Validate radial confinement + density guidance ---------------------------
# Measures each generated atom's transverse radius r in the per-candidate frame
# stored on test_set (the geometry the sampler used). 'shl' checks ALL atoms vs
# the two-sided band [r_lo, r_hi]. With v2 density guidance on, atoms should not
# just be in-band but CLUSTERED at the wall: the reconstructed template rho(r)
# (from the stored force table) is overlaid, and the generated wall thickness
# (std of r) is reported -- the key v2 metric. A/B: toggle DENSITY_MASKING in
# Section 4 and re-run Section 5 (off -> broad disk fill; on -> a narrow peak at
# the wall).
import numpy as np
import matplotlib.pyplot as plt

def atom_radii(frac_np, cell_np, d):
    ctr = d.tube_centroid[0].numpy(); a_hat = d.tube_a_hat[0].numpy()
    e1 = d.tube_e1[0].numpy(); e2 = d.tube_e2[0].numpy()
    cart = (frac_np % 1.0) @ cell_np
    z = cart @ a_hat
    rel = (cart - np.outer(z, a_hat)) - ctr
    return np.hypot(rel @ e1, rel @ e2)

def template_rho(d):
    """Reconstruct the template's rho(r) from the stored force f=d/dr log rho:
    rho(r) proportional to exp(cumtrapz(f) dr). Returns (grid, rho) or None."""
    f = d.dens_force[0].numpy()
    if not np.any(f):
        return None
    dr = float(d.dens_grid_dr); lo = float(d.dens_grid_lo)
    grid = lo + dr * np.arange(f.size)
    log_rho = np.concatenate([[0.0], np.cumsum(0.5 * (f[1:] + f[:-1]) * dr)])
    rho = np.exp(log_rho - log_rho.max())
    return grid, rho

pooled, rows = [], []
start = 0
for i in range(num_atoms.shape[0]):
    n_i = int(num_atoms[i])
    frac_i = frac_coords[start:start + n_i].numpy()
    cell_i = lattices[i].numpy()
    start += n_i
    d = test_set[i]
    if float(d.is_alx) <= 0:            # only real-template 'shl' carries a band
        continue
    r_chk = atom_radii(frac_i, cell_i, d)              # shl pins no atoms -> check all
    r_lo = float(d.r_min); r_hi = float(d.r_max) * (1.0 + CYL_MARGIN)
    inside = float(np.mean((r_chk >= r_lo) & (r_chk <= r_hi)))
    pooled.append((r_chk, r_lo, r_hi))
    rows.append((i, r_chk.size, r_lo, r_hi, float(r_chk.std()), inside))

if not rows:
    print('No atoms to validate (increase NUM_BATCHES, or check SC_TYPE=shl).')
else:
    print(f'{"#":>3} {"nchk":>4} {"r_lo":>6} {"r_hi":>6} {"wall_std":>8} {"inside":>7}')
    print('-' * 44)
    for i, nchk, rlo, rhi, wstd, ins in rows:
        print(f'{i:3d} {nchk:4d} {rlo:6.2f} {rhi:6.2f} {wstd:8.2f} {ins:6.0%}')

    fig, (ax, axp) = plt.subplots(1, 2, figsize=(12.5, 4.4),
                                  gridspec_kw={'width_ratios': [1.5, 1]},
                                  subplot_kw=None)
    axp.remove()
    axp = fig.add_subplot(1, 2, 2, projection='polar')
    all_r = np.concatenate([r for r, _, _ in pooled])
    ax.hist(all_r, bins=24, color='#43AA8B', edgecolor='white', density=True,
            label='all atoms')
    ax.axvline(np.median([rhi for _, _, rhi in pooled]), color='#F94144', ls='--',
               lw=2, label='median r_hi')
    ax.axvline(np.median([rlo for _, rlo, _ in pooled]), color='#277DA1', ls='--',
               lw=2, label='median r_lo')
    # Overlay the template rho(r) target (guidance goal) for candidate 0 if available.
    if bool(DENSITY_MASKING):
        tr = template_rho(test_set[rows[0][0]])
        if tr is not None:
            g, rho = tr
            ax.plot(g, rho * (ax.get_ylim()[1] * 0.9), color='#F8961E', lw=2.2,
                    label='template rho(r) target')
    ax.set_xlabel('transverse radius r (A)'); ax.set_ylabel('density')
    ax.set_title(f'Radial distribution  (SC_TYPE={SC_TYPE}, CYL={CYL_MASKING}, '
                 f'DENSITY={DENSITY_MASKING})')
    ax.legend(fontsize=8)

    # --- angular panel: where the atoms sit AROUND the tube ------------------
    # The radial histogram on the left cannot see azimuthal clustering: a wall
    # occupying a single 40-degree arc gives exactly the same r-distribution as a
    # complete ring. This polar view makes that failure mode obvious.
    _th_all = []
    for i, _, _, _, _, _ in rows:
        d = test_set[i]
        if float(getattr(d, 'is_alx', 0)) <= 0:
            continue
        n_i = int(num_atoms[i])
        off = int(num_atoms[:i].sum())
        cart_i = (frac_coords[off:off + n_i].numpy() % 1.0) @ lattices[i].numpy()
        ah = d.tube_a_hat[0].numpy(); ctr = d.tube_centroid[0].numpy()
        rel = (cart_i - np.outer(cart_i @ ah, ah)) - ctr
        u_i, v_i = rel @ d.tube_e1[0].numpy(), rel @ d.tube_e2[0].numpy()
        m = np.hypot(u_i, v_i) > 1e-6
        axp.scatter(np.arctan2(v_i[m], u_i[m]), np.hypot(u_i, v_i)[m], s=16,
                    alpha=0.65, edgecolors='none')
        _th_all.append(np.arctan2(v_i[m], u_i[m]))
    if _th_all:
        _th_all = np.concatenate(_th_all)
        _R = float(abs(np.exp(1j * _th_all).mean()))
        _z = _R * np.sqrt(_th_all.size)
        axp.set_title(f'Angular distribution\npooled ang_R={_R:.3f}  '
                      f'ang_z={_z:.2f}  (0.89 = uniform)', fontsize=10)
    else:
        axp.set_title('Angular distribution (no band)', fontsize=10)
    axp.set_ylim(0, None)
    axp.tick_params(labelsize=7)

    plt.tight_layout(); plt.show()
    print(f'\nAtoms within [r_lo, r_hi]: {np.mean([r[-1] for r in rows]):.0%} (structure-mean)')
    print(f'Wall thickness (mean std of r): {np.mean([r[4] for r in rows]):.2f} A  '
          f'-- smaller = thinner wall (lower BANDWIDTH_SCALE / higher DENSITY_STRENGTH)')
    if len(_th_all):
        print(f'Angular spread: pooled ang_R={_R:.3f} (0=around the whole tube, 1=one arc); '
              f'ang_z={_z:.2f} vs 0.89 for uniform-random. High values mean the wall is '
              f'only partly filled -- check TRACK_FRAME is on.')

---
## 6. Inspect the results

Convert the raw diffusion outputs (fractional coords, atom types, lattices) to
`pymatgen` `Structure` objects, then take a quick 3D look at one candidate.


In [ ]:
from pymatgen.core.lattice import Lattice
from pymatgen.core.structure import Structure


def lattices_to_params(lat):
    """(3,3) lattice matrix -> (lengths[3], angles_deg[3])."""
    lengths = np.linalg.norm(lat, axis=1)
    angles = np.zeros(3)
    for i in range(3):
        j, k = (i + 1) % 3, (i + 2) % 3
        cos = np.dot(lat[j], lat[k]) / (lengths[j] * lengths[k])
        angles[i] = np.degrees(np.arccos(np.clip(cos, -1.0, 1.0)))
    return lengths, angles


structures = []
start = 0
for i in range(num_atoms.shape[0]):
    n_i = int(num_atoms[i])
    coords_i = frac_coords[start:start + n_i].numpy()
    types_i = atom_types[start:start + n_i].numpy()
    start += n_i
    lengths_i, angles_i = lattices_to_params(lattices[i].numpy())
    species = [chemical_symbols[int(t)] for t in types_i]
    try:
        structure = Structure(
            Lattice.from_parameters(*lengths_i, *angles_i),
            species, coords_i, coords_are_cartesian=False)
        structures.append(structure)
        print(f'  [{i}] {structure.composition.reduced_formula}: '
              f'{n_i} atoms, a={lengths_i[0]:.2f} b={lengths_i[1]:.2f} c={lengths_i[2]:.2f} A')
    except Exception as e:
        structures.append(None)
        print(f'  [{i}] conversion failed ({e})')

print(f'\n{sum(s is not None for s in structures)} / {len(structures)} structures converted')


In [ ]:
# --- Per-candidate summary table ---------------------------------------------
# One row per generated candidate. Geometry is measured on the GENERATED atoms in
# the SAME per-candidate tube frame the sampler used (test_set[i]); r_min/r_max are
# the TEMPLATE band the mask enforced. `tmpl`/`tag` are the drawn template's row in
# nanotube_templates.npz and its provenance (now recorded by gen_utils).
import numpy as np, pandas as pd, torch
from scipy.spatial import cKDTree
from pymatgen.core.composition import Composition

_MARGIN = float(globals().get('CYL_MARGIN', 0.1))
_TAG = {1: 'real', 0: 'synthetic', -1: '—'}

def _min_nn(cart, axis_vec):
    "Min interatomic distance (A), tiling +/-1 image along the tube axis when known."
    if len(cart) < 2:
        return np.nan
    pts = np.vstack([cart + s * axis_vec for s in (-1, 0, 1)]) if axis_vec is not None else cart
    dd, _ = cKDTree(pts).query(cart, k=2)
    return float(dd[:, 1].min())

_ANG_KEYS = ('ang_R', 'ang_R2', 'ang_z', 'gap_max')


def _ang_stats(u, v, r_eps=1e-6):
    """Circular-uniformity statistics for transverse coords (u, v).

    ang_R  : circular resultant |mean(exp(i*theta))| -- 0 uniform, 1 all one angle.
    ang_R2 : same for the doubled angle; catches 2-lobe / antipodal structure.
             Real tubes sit near 0.10 (genuine n-fold symmetry), NOT 0.
    ang_z  : ang_R * sqrt(n). ang_R alone is not comparable across candidates --
             n random points already give E[R] ~ 0.886/sqrt(n) (0.14 at n=40,
             0.078 at n=128), so ang_z ~ 0.886 is the 'uniform' reference at ANY n.
    gap_max: largest empty angular sector, degrees.
    """
    r = np.hypot(u, v)
    m = r > r_eps
    th = np.arctan2(v[m], u[m])
    n = th.size
    if n < 2:
        return {k: np.nan for k in _ANG_KEYS}
    R = float(abs(np.exp(1j * th).mean()))
    s = np.sort(th % (2 * np.pi))
    gap = float(np.rad2deg(np.diff(np.concatenate([s, [s[0] + 2 * np.pi]])).max()))
    return dict(ang_R=R, ang_R2=float(abs(np.exp(2j * th).mean())),
                ang_z=R * np.sqrt(n), gap_max=gap)


def _template_ang(t_idx):
    """Same statistics for the TEMPLATE this candidate was drawn from -- the
    correct target. Returns (ang_R, gap_max) or (nan, nan)."""
    try:
        from gen_utils import NANOTUBE_DB
        from sc_utils import estimate_template_bounds
        t_frac, t_cell = NANOTUBE_DB.get(int(t_idx))
        if t_frac is None:
            return np.nan, np.nan
        tb = estimate_template_bounds(t_frac, t_cell)
        t_cart = (np.asarray(t_frac, float) % 1.0) @ np.asarray(t_cell, float)
        tz = t_cart @ tb['a_hat']
        trel = (t_cart - np.outer(tz, tb['a_hat'])) - tb['centroid']
        st = _ang_stats(trel @ tb['e1'], trel @ tb['e2'])
        return st['ang_R'], st['gap_max']
    except Exception:
        return np.nan, np.nan


_rows, _start = [], 0
for i in range(int(num_atoms.shape[0])):
    n_i = int(num_atoms[i])
    frac_i = frac_coords[_start:_start + n_i].numpy() % 1.0
    types_i = atom_types[_start:_start + n_i].numpy()
    cell_i = lattices[i].numpy()
    _start += n_i
    cart_i = frac_i @ cell_i

    if i < len(structures) and structures[i] is not None:
        formula = structures[i].composition.reduced_formula
    else:
        zc = np.unique(types_i, return_counts=True)
        formula = Composition({chemical_symbols[int(z)]: int(c)
                               for z, c in zip(*zc)}).reduced_formula

    d = test_set[i]
    t_idx = int(d.template_index[0]) if hasattr(d, 'template_index') else -1
    t_src = int(d.template_source[0]) if hasattr(d, 'template_source') else -1
    has_band = float(getattr(d, 'is_alx', torch.zeros(1))[0]) > 0
    axis_vec = cell_i[int(d.tube_axis)] if has_band else None

    if has_band:
        ctr = d.tube_centroid[0].numpy(); a_hat = d.tube_a_hat[0].numpy()
        e1 = d.tube_e1[0].numpy(); e2 = d.tube_e2[0].numpy()
        z = cart_i @ a_hat
        rel = (cart_i - np.outer(z, a_hat)) - ctr
        u, v = rel @ e1, rel @ e2
        r = np.hypot(u, v)
        r_lo, r_hi = float(d.r_min), float(d.r_max) * (1 + _MARGIN)
        C = np.cov(np.c_[u, v].T); w = np.clip(np.linalg.eigvalsh(C), 1e-9, None)
        geo = dict(r_min=float(d.r_min), r_max=float(d.r_max),
                   band_w=float(d.r_max) - float(d.r_min),
                   wall_std=float(r.std()),
                   pct_in=100.0 * float(np.mean((r >= r_lo) & (r <= r_hi))),
                   ellip=float(np.sqrt(w[1] / w[0])),
                   axial_c=float(np.linalg.norm(axis_vec)))
        # Angular uniformity around the tube -- the metric that detects atoms
        # bunched on one arc. Nothing else in this table sees it: pct_in and
        # wall_std are purely radial, and a perfect one-arc cluster can score
        # 100% / 0.00 on both.
        geo.update(_ang_stats(u, v))
    else:
        geo = dict(r_min=np.nan, r_max=np.nan, band_w=np.nan, wall_std=np.nan,
                   pct_in=np.nan, ellip=np.nan, axial_c=np.nan,
                   **{k: np.nan for k in _ANG_KEYS})

    _tR, _tgap = _template_ang(t_idx) if (has_band and t_idx >= 0) else (np.nan, np.nan)
    _rows.append(dict(idx=i, formula=formula, N=n_i, nk=int(num_known.flatten()[i]),
                      **geo, min_nn=_min_nn(cart_i, axis_vec),
                      ang_R_t=_tR, gap_t=_tgap,
                      tmpl=(t_idx if t_idx >= 0 else np.nan), tag=_TAG.get(t_src, '—')))

summary_df = pd.DataFrame(_rows)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')
print(summary_df.to_string(index=False))
print('\ncolumns: nk=pinned atoms (0 for shl) | r_min/r_max=template band (A) | '
      'band_w=r_max-r_min | wall_std=std of generated radii (A; thinner=better) | '
      'pct_in=% generated atoms inside [r_min, r_max*(1+margin)] | '
      'ellip=cross-section a/b (1=circular, >1.5 oval) | axial_c=tube period (A) | '
      'min_nn=min interatomic dist (A) | tmpl=nanotube_templates.npz row | tag=provenance')
print('angular: ang_R=circular resultant (0=spread around the tube, 1=all on one arc) | '
      'ang_R2=2-lobe resultant | ang_z=ang_R*sqrt(N), N-free (~0.89 = uniform-random) | '
      'gap_max=largest empty angular sector (deg) | ang_R_t/gap_t=the same for the '
      'TEMPLATE that candidate was drawn from -- the target to match')
print('targets (from the 2210-template survey): ang_R <= 0.06, gap_max <= 95 deg, '
      'ang_z <~ 1.5; ang_R2 should land NEAR the template value, not at 0 '
      '(real tubes have genuine n-fold structure at R2 ~ 0.10).')
if SC_TYPE == 'shl' and summary_df['tag'].eq('—').all():
    print('\n[note] template provenance not on test_set - re-run Section 5 after the '
          'gen_utils update so template_index/source are recorded.')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Which candidate to inspect (0 .. total_structures-1).
idx_to_inspect = 0

# --- Atoms in the RAW generated cell frame -----------------------------------
# Plot from frac @ lattices[idx] (NOT pymatgen's cart_coords): the band frame
# (centroid/axis/e1/e2) lives in this raw frame, whereas Lattice.from_parameters
# re-orients the cell, which would misalign the overlaid shell.
n     = int(num_atoms[idx_to_inspect])
start = int(num_atoms[:idx_to_inspect].sum())
cell  = lattices[idx_to_inspect].numpy()
frac  = frac_coords[start:start + n].numpy() % 1.0
cart  = frac @ cell
zs    = atom_types[start:start + n].numpy()          # atomic numbers (colour)
nk    = int(num_known.flatten()[idx_to_inspect])     # pinned skeleton size (0 for shl)

# --- Encoded skeleton geometry: the radial band + tube axis (if any) ----------
band = None
try:
    d = test_set[idx_to_inspect]
    if float(d.is_alx) > 0:
        margin = float(globals().get('CYL_MARGIN', 0.1))
        band = dict(ctr=d.tube_centroid[0].numpy(), a=d.tube_a_hat[0].numpy(),
                    e1=d.tube_e1[0].numpy(), e2=d.tube_e2[0].numpy(),
                    r_lo=float(d.r_min), r_hi=float(d.r_max) * (1.0 + margin))
except (NameError, AttributeError, IndexError):
    pass

def cyl_rings(band, r, cart, n_rings=6, n_theta=60):
    """Coaxial cylinder of radius r spanning the atoms' axial range (list of rings)."""
    a, ctr, e1, e2 = band['a'], band['ctr'], band['e1'], band['e2']
    zc = cart @ a
    th = np.linspace(0, 2 * np.pi, n_theta)
    ring = lambda z: (ctr + z * a) + r * np.outer(np.cos(th), e1) + r * np.outer(np.sin(th), e2)
    return [ring(z) for z in np.linspace(zc.min(), zc.max(), n_rings)]

def axis_angles(a):
    """(elev, azim) that look DOWN the tube axis -> the tube's cross-section."""
    return np.degrees(np.arcsin(np.clip(a[2], -1, 1))), np.degrees(np.arctan2(a[1], a[0]))

if band is not None:
    dax = axis_angles(band['a'])                                 # down the axis
    prof = (0.0, np.degrees(np.arctan2(band['a'][1], band['a'][0])) + 90.0)  # along the axis
else:
    dax, prof = (90, -90), (0, -90)

views = [('Perspective', 22, -60), ('Front (xz)', 0, -90), ('Side (yz)', 0, 0),
         ('Top (xy)', 90, -90), ('Down tube axis', *dax), ('Axis profile', *prof)]

# Common cube so every panel has TRUE equal aspect (a tube must look like a tube).
# Center/size on the ENCODED axis + r_hi, not the generated atoms: if the atoms
# under-fill or sit off-center in the band, this keeps the true tube axis centered
# and the full shell visible instead of auto-fitting to wherever the atoms landed.
if band is not None:
    mid = band['ctr']
    half = max(band['r_hi'], float((cart.max(0) - cart.min(0)).max()) / 2) + 1.0
else:
    mid = cart.mean(0)
    half = float((cart.max(0) - cart.min(0)).max()) / 2 + 1.0

fig = plt.figure(figsize=(16, 9))
s0 = structures[idx_to_inspect] if idx_to_inspect < len(structures) else None
formula = s0.composition.reduced_formula if s0 is not None else f'structure {idx_to_inspect}'
for k, (title, elev, azim) in enumerate(views):
    ax = fig.add_subplot(2, 3, k + 1, projection='3d')
    ax.scatter(cart[:, 0], cart[:, 1], cart[:, 2], c=zs, cmap='viridis', s=55,
               edgecolors='k', linewidths=0.3)
    if nk > 0:   # ring any pinned skeleton atoms in red (shl pins none)
        ax.scatter(cart[:nk, 0], cart[:nk, 1], cart[:nk, 2], s=160, facecolors='none',
                   edgecolors='red', linewidths=1.6)
    if band is not None:   # draw the encoded shell: inner (r_lo) + outer (r_hi) cylinders
        for r, col in [(band['r_lo'], '#277DA1'), (band['r_hi'], '#F94144')]:
            if r <= 0:
                continue
            for pts in cyl_rings(band, r, cart):
                ax.plot(pts[:, 0], pts[:, 1], pts[:, 2], color=col, lw=0.8, alpha=0.45)
    ax.set_title(title, fontsize=10)
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlim(mid[0] - half, mid[0] + half)
    ax.set_ylim(mid[1] - half, mid[1] + half)
    ax.set_zlim(mid[2] - half, mid[2] + half)
    ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
    try:
        ax.set_box_aspect((1, 1, 1))
    except Exception:
        pass

if nk > 0:
    skel = f'{nk} pinned atoms (red rings) + band shell'
elif band is not None:
    skel = 'radial band shell (blue=r_lo inner, red=r_hi outer)'
else:
    skel = 'no geometric skeleton'
fig.suptitle(f'{formula}  —  {n} atoms  —  encoded skeleton: {skel}',
             y=0.99, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6b. Diffusion trajectory (how a candidate forms)

`sample_scigen` returns a per-step trajectory as its second value (`traj`). The
cells below visualise structure 0 evolving from noise (t=1.0) to the final
crystal (t=0.0). Under `shl` **no atoms are pinned** — every atom is generated
from noise. What stays fixed is the tube's *geometry*: the soft radial band and
`ψ(t)`-ramped density guidance progressively pull the diffusing atoms onto the
wall shell, so the cloud condenses from a diffuse blob into a tube wall over the
trajectory. This is the visual signature of the geometric-shell constraint.

With `TRACK_FRAME` on, the drawn `r_min`/`r_max` cylinders are the band **as actually
enforced at that step**: they start small and grow with the cell (the `s=` factor in each
title), and the per-panel `ang_R` shows atoms spreading around the tube instead of riding
up one side. (The removed atom-pinning modes would have re-imposed a fixed skeleton at every
step; `shl` has none, so no atoms are ring-highlighted below.)

In [ ]:
# --- Diffusion trajectory: Perspective view per step ---------------------------
# For each diffusion step, show the Perspective view only, annotated with the
# encoded r_min/r_max band and the step's live x/y/z (Angstrom) bounding-box
# extents -- so you can watch the atom cloud condense from a diffuse blob into
# the tube's true footprint. Pick which candidate with idx_traj_inspect.
import numpy as np
import matplotlib.pyplot as plt
from sc_utils import chemical_symbols

assert 'traj' in globals(), (
    "`traj` not found — run the generation cell (Section 5) first. "
    "It is the 2nd return value of model.sample_scigen and holds the trajectory.")

traj_coords    = traj['all_frac_coords'].detach().cpu()   # (T+1, total_atoms, 3)
traj_lattices  = traj['all_lattices'].detach().cpu()      # (T+1, n_struct, 3, 3)
traj_types     = traj['atom_types'].detach().cpu()        # (T+1, total_atoms) 1-indexed Z
traj_num_atoms = traj['num_atoms'].detach().cpu()
traj_num_known = traj['num_known'].detach().cpu().flatten()

# ===== SELECT WHICH CANDIDATE TO INSPECT =====
idx_traj_inspect = 0  # Change to inspect a different candidate (0 .. total_structures-1)
# =============================================

n_steps = traj_coords.shape[0]
na_cand = int(traj_num_atoms[idx_traj_inspect].item())
nk_cand = int(traj_num_known[idx_traj_inspect].item()) if traj_num_known.numel() > 0 else 0

n_frames = min(10, n_steps)
frame_indices = np.linspace(0, n_steps - 1, n_frames, dtype=int)
print(f'Trajectory: {n_steps} steps, showing {n_frames} frames')
print(f'Candidate {idx_traj_inspect}: {na_cand} atoms (num_known={nk_cand}; shl pins none)')

def frame_to_cart(step_idx, cand_idx):
    """Cartesian coords + per-atom Z for candidate cand_idx at step step_idx."""
    a_start = int(num_atoms[:cand_idx].sum().item())
    n_cand = int(traj_num_atoms[cand_idx].item())
    frac = traj_coords[step_idx, a_start:a_start + n_cand].numpy() % 1.0
    lat  = traj_lattices[step_idx, cand_idx].numpy()
    zs   = traj_types[step_idx, a_start:a_start + n_cand].int().tolist()
    return frac @ lat, zs

# --- Encoded skeleton geometry: the radial band + tube axis (if any) ----------
band = None
try:
    d = test_set[idx_traj_inspect]
    if float(d.is_alx) > 0:
        margin = float(globals().get('CYL_MARGIN', 0.1))
        _lat0 = d.lattice_known[0].numpy()
        _k = int(d.tube_axis); _o0, _o1 = [j for j in range(3) if j != _k]
        _ah0 = _lat0[_k] / np.linalg.norm(_lat0[_k])
        _p00 = _lat0[_o0] - (_lat0[_o0] @ _ah0) * _ah0
        _p10 = _lat0[_o1] - (_lat0[_o1] @ _ah0) * _ah0
        band = dict(ctr=d.tube_centroid[0].numpy(), a=d.tube_a_hat[0].numpy(),
                    e1=d.tube_e1[0].numpy(), e2=d.tube_e2[0].numpy(),
                    r_lo=float(d.r_min), r_hi=float(d.r_max) * (1.0 + margin),
                    # reference quantities so band_at() can rescale per frame
                    ctr_frac=getattr(d, 'tube_centroid_frac',
                                     torch.full((1, 3), 0.5))[0].numpy(),
                    area_ref=float(abs(np.cross(_lat0[_o0], _lat0[_o1]) @ _ah0)),
                    pn_ref=(float(np.linalg.norm(_p00)), float(np.linalg.norm(_p10))))
except (NameError, AttributeError, IndexError):
    pass

def band_at(step_idx, cand_idx):
    """Band + frame as the SAMPLER sees them at this step.

    With TRACK_FRAME the sampler rebuilds the frame from the current lattice and
    scales the band by the transverse cell scale s_t, so a fixed cylinder would be
    badly wrong early on (the cell starts ~1 A and grows to ~14 A). Mirrors
    diff_utils.lattice_tube_frame / transverse_scale in numpy.
    """
    if band is None:
        return None
    lat = traj_lattices[step_idx, cand_idx].numpy()
    if not bool(globals().get('TRACK_FRAME', True)):
        return dict(band, s=1.0)
    k = int(test_set[cand_idx].tube_axis)
    o0, o1 = [j for j in range(3) if j != k]
    a = lat[k]
    a_hat = a / max(np.linalg.norm(a), 1e-8)
    p0 = lat[o0] - (lat[o0] @ a_hat) * a_hat
    p1 = lat[o1] - (lat[o1] @ a_hat) * a_hat
    e1 = p0 / max(np.linalg.norm(p0), 1e-8)
    e2 = np.cross(a_hat, e1)
    f = band['ctr_frac']
    ctr = f[o0] * p0 + f[o1] * p1
    area = abs(np.cross(lat[o0], lat[o1]) @ a_hat)
    s_area = np.sqrt(max(area / max(band['area_ref'], 1e-8), 0.0))
    s_lin = 0.5 * (np.linalg.norm(p0) / max(band['pn_ref'][0], 1e-8)
                   + np.linalg.norm(p1) / max(band['pn_ref'][1], 1e-8))
    s = float(np.clip(np.clip(s_area, 0.25 * s_lin, 4.0 * s_lin), 1e-3, 1e3))
    return dict(ctr=ctr, a=a_hat, e1=e1, e2=e2,
                r_lo=band['r_lo'] * s, r_hi=band['r_hi'] * s, s=s)


def cyl_rings(band, r, cart, n_rings=6, n_theta=60):
    """Coaxial cylinder of radius r spanning the atoms' axial range (list of rings)."""
    a, ctr, e1, e2 = band['a'], band['ctr'], band['e1'], band['e2']
    zc = cart @ a
    th = np.linspace(0, 2 * np.pi, n_theta)
    ring = lambda z: (ctr + z * a) + r * np.outer(np.cos(th), e1) + r * np.outer(np.sin(th), e2)
    return [ring(z) for z in np.linspace(zc.min(), zc.max(), n_rings)]

# Pre-fetch all frames once so the box can be sized to fit the WHOLE trajectory:
# a single shared, band-centered box (not re-centered per step on wherever the
# still-diffuse cloud happens to sit) makes the condensing motion legible.
frames = [frame_to_cart(s, idx_traj_inspect) for s in frame_indices]
all_cart = np.concatenate([c for c, _ in frames], axis=0)
if band is not None:
    mid = band['ctr']
    half = max(band['r_hi'], float((all_cart.max(0) - all_cart.min(0)).max()) / 2) + 1.0
else:
    mid = all_cart.mean(0)
    half = float((all_cart.max(0) - all_cart.min(0)).max()) / 2 + 1.0

ncols = min(5, n_frames)
nrows = int(np.ceil(n_frames / ncols))
fig = plt.figure(figsize=(3.6 * ncols, 3.8 * nrows))

for step_num, step_idx in enumerate(frame_indices):
    cart, zs = frames[step_num]
    d_ext = cart.max(0) - cart.min(0)   # live x/y/z (Angstrom) bounding-box extent

    ax = fig.add_subplot(nrows, ncols, step_num + 1, projection='3d')
    ax.scatter(cart[:, 0], cart[:, 1], cart[:, 2], c=zs, cmap='viridis', s=45,
               edgecolors='k', linewidths=0.3)
    if nk_cand > 0:   # ring any pinned skeleton atoms in red (shl pins none)
        ax.scatter(cart[:nk_cand, 0], cart[:nk_cand, 1], cart[:nk_cand, 2],
                   s=140, facecolors='none', edgecolors='red', linewidths=1.6)
    band_t = band_at(step_idx, idx_traj_inspect)
    if band_t is not None:   # the shell AS ENFORCED at this step (scales with the cell)
        for r, col in [(band_t['r_lo'], '#277DA1'), (band_t['r_hi'], '#F94144')]:
            if r <= 0:
                continue
            for pts in cyl_rings(band_t, r, cart):
                ax.plot(pts[:, 0], pts[:, 1], pts[:, 2], color=col, lw=0.7, alpha=0.4)

    ax.view_init(elev=22, azim=-60)   # Perspective only
    ax.set_xlim(mid[0] - half, mid[0] + half)
    ax.set_ylim(mid[1] - half, mid[1] + half)
    ax.set_zlim(mid[2] - half, mid[2] + half)
    ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
    try:
        ax.set_box_aspect((1, 1, 1))
    except Exception:
        pass

    t_frac = step_idx / max(n_steps - 1, 1)
    band_txt = ''
    if band_t is not None:
        rel_t = (cart - np.outer(cart @ band_t['a'], band_t['a'])) - band_t['ctr']
        u_t, v_t = rel_t @ band_t['e1'], rel_t @ band_t['e2']
        m_t = np.hypot(u_t, v_t) > 1e-6
        R_t = (abs(np.exp(1j * np.arctan2(v_t[m_t], u_t[m_t])).mean())
               if m_t.sum() > 1 else np.nan)
        band_txt = (f"r_min={band_t['r_lo']:.2f}  r_max={band_t['r_hi']:.2f} Å  "
                    f"(s={band_t['s']:.2f})\nang_R={R_t:.2f}  ")
    ax.set_title(f'Step {step_idx} (t={1-t_frac:.2f})\n{band_txt}'
                 f'x={d_ext[0]:.1f}  y={d_ext[1]:.1f}  z={d_ext[2]:.1f} Å',
                 fontsize=8, fontweight='bold')

fig.suptitle(f'Diffusion trajectory ({SC_TYPE}, candidate {idx_traj_inspect}): '
             f'noise → nanotube wall  [num_known={nk_cand}]  —  Perspective view',
             y=1.0, fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

---
## 7. Export

> **Deeper analysis moved:** lattice/space-group/XRD-style sanity checks and the
> geometry/chemistry/CHGNet-energy/relaxation screening pipeline now live in
> [`ntgen_validation.ipynb`](./ntgen_validation.ipynb) — run it against the CIFs
> exported below. See `docs/usage/inspecting-outputs.md`.

Save the candidates as CIF files, then download them as a zip archive
(Colab's disk is wiped when the runtime ends).


In [ ]:
output_dir = os.path.join(NOTEBOOK_DIR, 'generated_cifs')
os.makedirs(output_dir, exist_ok=True)

n_written = 0
for i, structure in enumerate(structures):
    if structure is None:
        continue
    formula = structure.composition.reduced_formula
    cif_path = os.path.join(output_dir, f'{SC_TYPE}_{formula}_{i:03d}.cif')
    structure.to(filename=cif_path, fmt='cif')
    print(f'Saved: {os.path.basename(cif_path)}')
    n_written += 1

print(f'\n{n_written} CIF files saved to {output_dir}')


In [ ]:
# --- Export per-candidate geometry sidecar (for ntgen_validation.ipynb) -------
# The tube frame + radial band live ONLY in memory on test_set[i]; they are NOT
# recoverable from the CIF (Lattice.from_parameters re-orients the cell, and for
# 'shl' the band / rho(r) were measured from template atoms that are discarded).
# So we write them alongside the CIFs, keyed by the SAME stem as each CIF, plus
# `lattice_raw` (the un-reoriented cell) so validation can rebuild atoms in the
# sampler's frame with `frac @ lattice_raw`.
import json

def _j(x):
    """Tensor/ndarray/scalar -> JSON-friendly python (nested list or number)."""
    import numpy as _np
    if hasattr(x, 'detach'):
        x = x.detach().cpu().numpy()
    if isinstance(x, _np.ndarray):
        return x.astype(float).tolist()
    return x

meta_records = []
num_known_flat = num_known.flatten()
for i in range(num_atoms.shape[0]):
    s = structures[i] if i < len(structures) else None
    if s is None:                                  # no CIF was written -> no sidecar row
        continue
    formula = s.composition.reduced_formula
    stem = f'{SC_TYPE}_{formula}_{i:03d}'          # MUST match the CIF filename stem
    d = test_set[i]
    is_alx = float(d.is_alx)                        # legacy flag: 1.0 = real Alexandria band active
    rec = {
        'stem': stem,
        'index': i,
        'sc_type': SC_TYPE,
        'formula': formula,
        'num_atoms': int(num_atoms[i]),
        'num_known': int(num_known_flat[i]),
        'is_alx': is_alx,
        'cyl_margin': float(CYL_MARGIN),
        'lattice_raw': _j(lattices[i]),            # raw cell BEFORE from_parameters re-orient
    }
    if is_alx > 0:                                 # real-template shl carries a tube frame + band
        rec.update({
            'tube_axis': int(d.tube_axis),
            'tube_centroid': _j(d.tube_centroid[0]),
            'tube_centroid_frac': _j(getattr(d, 'tube_centroid_frac', [[0.5, 0.5, 0.5]])[0]),
            'tube_a_hat': _j(d.tube_a_hat[0]),
            'tube_e1': _j(d.tube_e1[0]),
            'tube_e2': _j(d.tube_e2[0]),
            'r_min': float(d.r_min),
            'r_max': float(d.r_max),
            'dens_grid_lo': float(d.dens_grid_lo),
            'dens_grid_dr': float(d.dens_grid_dr),
            'dens_force': _j(d.dens_force[0]),
        })
    else:                                          # van / synthetic-ring fallback: no band
        rec.update({k: None for k in (
            'tube_axis', 'tube_centroid', 'tube_a_hat', 'tube_e1', 'tube_e2',
            'r_min', 'r_max', 'dens_grid_lo', 'dens_grid_dr', 'dens_force')})
    meta_records.append(rec)

meta_path = os.path.join(output_dir, 'candidates_metadata.json')
with open(meta_path, 'w') as fh:
    json.dump(meta_records, fh, indent=2)
print(f'Wrote {len(meta_records)} geometry records -> {meta_path}')
print('  (ntgen_validation.ipynb joins these to the CIFs on the "stem" key)')


In [ ]:
# Download CIF files as a zip archive (Colab only)
try:
    import shutil
    from google.colab import files
    zip_name = f'ntgen_{SC_TYPE}_cifs'
    zip_path = shutil.make_archive(os.path.join('/content', zip_name), 'zip', output_dir)
    files.download(zip_path)
    print(f'Downloading {zip_name}.zip')
except ImportError:
    print('Not running in Colab — skipping download.')
except Exception as e:
    print(f'Download failed: {e}')
    print(f'CIF files are available at: {output_dir}')


---
## 8. Try it yourself

Go back to **Section 4** and change the parameters, then re-run from Section 5:

- **More structures:** increase `BATCH_SIZE` or `NUM_BATCHES`
- **Geometric shell (de-novo):** set `SC_TYPE='shl'` to pin only a real tube's
  *shape* and generate every atom fresh inside the wall band (`num_known=0`).
  Watch the validation cell (Section 5): with `CYL_MASKING=True` the generated
  atoms should collapse onto `[r_min, r_max]`; with it `False` they fill the box.
- **Thin walls (density guidance, v2):** with `DENSITY_MASKING=True` the atoms
  concentrate onto the real wall instead of filling the disk. Lower
  `BANDWIDTH_SCALE` (e.g. 0.4–0.6) or raise `DENSITY_STRENGTH` for a thinner
  wall; the validation cell reports the wall thickness (std of r) and overlays
  the template ρ(r) target. The "Down tube axis" panel (Section 6) should shift
  from a filled disk to a ring.
- **Unconstrained baseline:** set `SC_TYPE='van'` for plain unconditional
  generation (no tube constraint) to compare against the shell
- **Different step size:** try `STEP_LR = 1e-5` and compare structure quality

### Notes

- **`shl` at a glance**: `shl` pins **no atoms**, only the geometry (cell +
  radial band), and generates the tube's full real atom count fresh — maximal
  chemistry freedom, tube shape enforced by the soft radial band. Check the
  Section-5 printout: `shl` shows `num_known=0` with a `shell band [r_min, r_max]`.
- **Atoms bunched on one arc?** Check `TRACK_FRAME=True` first — a fixed frame is the
  root cause, and `ang_R` / `gap_max` in the Section 6 table are what measure it
  (`pct_in` and `wall_std` are purely radial and score a one-arc wall as perfect).
  Compare against the `ang_R_t` / `gap_t` columns, which are the drawn template's own
  values. Only if `ang_R` is still high should you enable `ANG_SPREAD`.
- **Never raise `PSI_START` above 0.0.** `ψ(t)` gates the geometric forces as well as
  the mask, so `PSI_START=1.0` fires the full-strength radial band at the ~1 Å noise
  cell (measured: `pct_in` 100 % → 67 %). Use `GEOM_PIN_SCHEDULE` to ramp the forces
  separately if you want a hard-pinned lattice.
- **Band vs density guidance**: the **band** (`CYL_MASKING`) *bounds* atoms to
  the shell; the **density guidance** (`DENSITY_MASKING`, v2) *shapes* them into
  a thin wall within it by following the template's empirical ρ(r). Use both for
  a realistic single-wall cross-section; the band alone leaves a filled disk.
- **The model is still mp_20** *(important)*: geometry (the pinned cell / band /
  wall) renders faithfully, but mp_20 was trained on ≤20-atom bulk crystals with
  no vacuum/tube, so at ~42 atoms it cannot place atoms/chemistry ideally. For
  genuinely good results, retrain on the Alexandria 1D set — see `RETRAIN_ALX.md`.
- **Decorator chemistry** *(#4)*: decoded species are restricted to a plausible set
  (`DECORATOR_DISALLOWED_Z` drops At/Rn/Fr/Ra and transuranics incl. Fm). Always run the
  screening pipeline (`script/eval_screen.py`) before trusting any candidate.
- **Fallback**: a `shl` draw with no fitting template silently uses the private
  synthetic ring (`_SC_NanotubeFallback`) — check the Section-5 printout to see
  what was applied.

### References

- **SCIGEN:** Okabe et al., "Structural constraint integration in a generative
  model for the discovery of quantum materials," *Nature Materials* (2025).
  [DOI](https://doi.org/10.1038/s41563-025-02355-y) |
  [GitHub](https://github.com/RyotaroOKabe/SCIGEN)
- **Alexandria database:** Schmidt et al., "Machine-learning-assisted
  determination of the global zero-temperature phase diagram of materials,"
  and the Alexandria 1D dataset. [alexandria.icams.rub.de](https://alexandria.icams.rub.de/)
- **pymatgen:** Ong et al., "Python Materials Genomics (pymatgen)," *Comput.
  Mater. Sci.* 68, 314-319 (2013).
  [GitHub](https://github.com/materialsproject/pymatgen)